In [ ]:
import json
import re
import spacy
from sklearn.cluster import DBSCAN
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

In [45]:
# Read the saved house deatil data: (Repalced by reading the data from the database)
with open("cur_data.json", "r") as data:
    home_data = json.load(data)

In [ ]:
# Convert all data to lowercase and delete all spaces
for key, value_list in home_data.items():
    home_data[key] = [item.lower().strip() for item in value_list]

In [ ]:
# Load NLP model
nlp = spacy.load("en_core_web_md")

# Initial category dictionary with sample data
categories = {
    "Amenities": ["fitness center", "swimming pool", "clubhouse"],
    "Pet Policy": ["dogs allowed", "pet washing station"],
    "Security": ["night patrol", "security cameras"]
}

def categorize_preference(preference):
    pref_vector = nlp(preference.lower())

    best_match = None
    best_similarity = 0.6  # Similarity threshold

    for category, examples in categories.items():
        avg_sim = sum(pref_vector.similarity(nlp(e)) for e in examples) / len(examples)

        if avg_sim > best_similarity:
            best_similarity = avg_sim
            best_match = category

    if best_match:
        categories[best_match].append(preference)
    else:
        unclassified_prefs.append(preference)  # Store for clustering

In [ ]:
def cluster_new_preferences(unclassified_prefs):
    if not unclassified_prefs:
        return

    vectors = np.array([nlp(pref).vector for pref in unclassified_prefs])

    # Clustering with DBSCAN (self-organizing)
    clustering = DBSCAN(eps=0.4, min_samples=1, metric="cosine").fit(vectors)

    cluster_dict = {}
    for i, label in enumerate(clustering.labels_):
        cluster_dict.setdefault(label, []).append(unclassified_prefs[i])

    # Assign new categories
    for cluster, prefs in cluster_dict.items():
        new_category = prefs[0]  # Name category based on the first preference
        categories[new_category] = prefs

    print(f"New categories created: {list(cluster_dict.values())}")

# Example: Adding new preferences dynamically
new_preferences = ["EV Charging Stations", "Granite Countertops", "Gated Community", "Bike Storage", "Smart Locks", "Co-working Space"]
unclassified_prefs = []

for pref in new_preferences:
    categorize_preference(pref)

# Cluster unclassified preferences to create new categories
cluster_new_preferences(unclassified_prefs)

print("Final Categories:", categories)